In [ ]:
import os
import json
import datetime
import re
from typing import List

from pydantic import BaseModel, Field, ValidationError

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

# ============================================================
# CONFIG
# ============================================================
TIMEZONE = "Asia/Kolkata"
MODEL = "llama-3.1-8b-instant"

# ============================================================
# Pydantic Models (SOURCE OF TRUTH)
# ============================================================
class ExtractedContext(BaseModel):
    start_date: str = Field(description="Start date in YYYY-MM-DD or 'today'")
    duration_days: int = Field(description="Number of days")
    tasks: List[str] = Field(description="Learning tasks")
    email: str | None = Field(default=None, description="User email")

class DailyTask(BaseModel):
    title: str
    start_time: str
    end_time: str

class Schedule(BaseModel):
    days: int
    daily_template: List[DailyTask]

# ============================================================
# GOOGLE CALENDAR
# ============================================================
def get_calendar_service():
    SCOPES = ["https://www.googleapis.com/auth/calendar"]
    creds = None

    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                "credentials.json", SCOPES
            )
            creds = flow.run_local_server(port=0)

        with open("token.json", "w") as token:
            token.write(creds.to_json())

    return build("calendar", "v3", credentials=creds)

# ============================================================
# 1️⃣ CONTEXT EXTRACTION (STRICT)
# ============================================================
def extract_context(user_request: str) -> ExtractedContext:
    parser = PydanticOutputParser(pydantic_object=ExtractedContext)

    prompt = ChatPromptTemplate.from_messages([
        ("system", """
            - You are a strict information extraction engine and also a data converter engine which understands tricky questions and converts them into a simple form that can be parsed by a JSON parser.
            - You can understand any tricky questions and convert them into a simple form that can be parsed by a JSON parser.
            - You can understand any grammer errors and convert them into a simple form that can be parsed by a JSON parser.
            - You can understand any spelling errors and convert them into a simple form that can be parsed by a JSON parser.
            - You are the first main model should not make any errors in the output.
            - The output should be taken in the proper format as per the schema.
            - And maintain the Strict Rules.

            Rules:
            - DO NOT guess
            - DO NOT infer missing values
            - Duration must be a number of days
            - Tasks must be learning-related
            - NO explanations
            - NO markdown
            - NO extra text
            - NO code
            - NO comments
            - Do NOT convert dates to ISO format here
            - Output ONLY valid JSON.

            Schema:
            {{
                "start_date": "YYYY-MM-DD",
                "duration_days": number,
                "tasks": [string],
                "email": string | null
            }}
        """),
        ("human", "{input}")
    ])

    llm = ChatGroq(model=MODEL, temperature=0)
    chain = prompt | llm

    max_attempts = 3
    for attempt in range(max_attempts):
        try:
            response = chain.invoke({"input": user_request})
            if response and response.content:
                data = json.loads(response.content)
                return ExtractedContext(**data)
        except Exception as e:
            print(f"⚠️ Context extraction attempt {attempt + 1} failed: {e}")

    raise ValueError("Context extraction failed after 3 attempts. Check your LLM response.")

# ============================================================
# 2️⃣ DATE NORMALIZATION
# ============================================================
def normalize_start_date(date_str: str) -> datetime.date:
    if re.match(r"\d{4}-\d{2}-\d{2}", date_str):
        return datetime.date.fromisoformat(date_str)

    ordinal = re.search(r"(\d{1,2})(st|nd|rd|th)", date_str.lower())
    if ordinal:
        day = int(ordinal.group(1))
        today = datetime.date.today()
        return datetime.date(today.year, today.month, day)

    raise ValueError("Unsupported date format")
# ============================================================
# 3️⃣ SCHEDULER (USES CONTEXT ONLY)
# ============================================================
def generate_schedule(context: ExtractedContext) -> Schedule:
    prompt = ChatPromptTemplate.from_messages([
        ("system", """
            You are a scheduler.

            Rules:
            - Use ONLY the provided tasks
            - Number of days MUST equal duration_days
            - Create ONE focused learning block per day
            - Output ONLY JSON
            - Time format HH:MM

            Schema:
            {{
            "days": number,
            "daily_template": [
                {{ "title": string, "start_time": "HH:MM", "end_time": "HH:MM" }}
            ]
            }}
        """),
        ("human", """
            Duration days: {days}
            Tasks: {tasks}
        """)
    ])

    llm = ChatGroq(model=MODEL, temperature=0)
    chain = prompt | llm

    response = chain.invoke({
        "days": context.duration_days,
        "tasks": context.tasks
    })

    try:
        data = json.loads(response.content)
        return Schedule(**data)
    except ValidationError as e:
        raise ValueError(f"Schedule validation failed: {e}")

# ============================================================
# 4️⃣ CALENDAR EVENT CREATION (NO LLM)
# ============================================================
def create_calendar_events(
    schedule: Schedule,
    context: ExtractedContext
):
    service = get_calendar_service()
    start_date = normalize_start_date(context.start_date)

    for day_offset in range(schedule.days):
        current_date = start_date + datetime.timedelta(days=day_offset)
        task = schedule.daily_template[day_offset]

        start_dt = datetime.datetime.combine(
            current_date,
            datetime.datetime.strptime(task.start_time, "%H:%M").time()
        )

        end_dt = datetime.datetime.combine(
            current_date,
            datetime.datetime.strptime(task.end_time, "%H:%M").time()
        )

        event = {
            "summary": task.title,
            "description": task.title,
            "start": {
                "dateTime": start_dt.isoformat(),
                "timeZone": TIMEZONE
            },
            "end": {
                "dateTime": end_dt.isoformat(),
                "timeZone": TIMEZONE
            }
        }

        if context.email:
            event["attendees"] = [{"email": context.email}]

        service.events().insert(
            calendarId="primary",
            body=event,
            sendUpdates="all"
        ).execute()

        print(f"✅ Created: {task.title} on {current_date}")


e:\git_projects\Projects\AI Productivity Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 